<a href="https://colab.research.google.com/github/dwatd/koreanVocabulary/blob/main/trying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install konlpy
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
from pandas import DataFrame
from konlpy.tag import Okt
from collections import Counter

In [148]:
text = """한강은 서울 도심을 지나가는 강입니다. 옛날 사람들은 한강 물을 마시고 한강에서 낚시도 하면서 살았습니다. 그리고 기차나 차가 없을 때 한강에서 배를 타고 다른 지역으로 갔습니다. 긴 역사 속에서 한강은 한국 사람들에게 중요한 강이었습니다."""
text = re.sub(r'[^\w\s]', '', text)

In [149]:
okt = Okt()
tokens = okt.pos(text, norm=True, stem=True)
print(tokens)

[('한강', 'Noun'), ('은', 'Josa'), ('서울', 'Noun'), ('도심', 'Noun'), ('을', 'Josa'), ('지나가다', 'Verb'), ('강', 'Noun'), ('이다', 'Adjective'), ('옛날', 'Noun'), ('사람', 'Noun'), ('들', 'Suffix'), ('은', 'Josa'), ('한강', 'Noun'), ('물', 'Noun'), ('을', 'Josa'), ('말다', 'Verb'), ('한강', 'Noun'), ('에서', 'Josa'), ('낚시', 'Noun'), ('도', 'Josa'), ('하다', 'Verb'), ('살다', 'Verb'), ('그리고', 'Conjunction'), ('기차', 'Noun'), ('나', 'Josa'), ('차갑다', 'Adjective'), ('없다', 'Adjective'), ('때', 'Noun'), ('한강', 'Noun'), ('에서', 'Josa'), ('배', 'Noun'), ('를', 'Josa'), ('타고', 'Noun'), ('다른', 'Noun'), ('지역', 'Noun'), ('으로', 'Josa'), ('가다', 'Verb'), ('기다', 'Verb'), ('역사', 'Noun'), ('속', 'Noun'), ('에서', 'Josa'), ('한강', 'Noun'), ('은', 'Josa'), ('한국', 'Noun'), ('사람', 'Noun'), ('들', 'Suffix'), ('에게', 'Josa'), ('중요하다', 'Adjective'), ('강', 'Noun'), ('이다', 'Verb')]


In [150]:
filtered_tokens = []
# Filtering (left only Noun, Verb and Adjective)
for word, pos in tokens:
  if pos == 'Noun' or pos == 'Verb' or pos == 'Adjective':
    filtered_tokens.append(word)
print(filtered_tokens)

['한강', '서울', '도심', '지나가다', '강', '이다', '옛날', '사람', '한강', '물', '말다', '한강', '낚시', '하다', '살다', '기차', '차갑다', '없다', '때', '한강', '배', '타고', '다른', '지역', '가다', '기다', '역사', '속', '한강', '한국', '사람', '중요하다', '강', '이다']


In [151]:
counted_tokens = Counter(filtered_tokens)
print(counted_tokens)

Counter({'한강': 5, '강': 2, '이다': 2, '사람': 2, '서울': 1, '도심': 1, '지나가다': 1, '옛날': 1, '물': 1, '말다': 1, '낚시': 1, '하다': 1, '살다': 1, '기차': 1, '차갑다': 1, '없다': 1, '때': 1, '배': 1, '타고': 1, '다른': 1, '지역': 1, '가다': 1, '기다': 1, '역사': 1, '속': 1, '한국': 1, '중요하다': 1})


In [152]:
my_words_df = DataFrame.from_records(data=counted_tokens.most_common(), columns=['word', 'frequency'])

In [138]:
from google.colab import files
dataset = files.upload()
df_topik = pd.read_csv('results.tsv', sep='\t')
print(df_topik.head())

Saving results.tsv to results (3).tsv
     rank  word part_of_speech hanja explanation nikl_level topik_level
0     NaN  -가13             접사   NaN         전문가         중급         NaN
1  1195.0    가게             명사   NaN        에 가다         초급           A
2   898.0  가격03             명사    價格       이 비싸다         초급           B
3  2986.0  가구03             명사    家口         NaN        NaN           C
4  7434.0  가구04             명사    家具          책상         초급           B


In [153]:
df_topik['word'] = df_topik['word'].str.replace(r'\d+', '', regex=True)
df_topik = df_topik.drop_duplicates(subset=['word'])

In [154]:
final_df = pd.merge(my_words_df, df_topik, on='word', how='left').drop(columns=['rank', 'hanja', 'explanation', 'nikl_level'])
print(final_df)

    word  frequency part_of_speech topik_level
0     한강          5          고유 명사           A
1      강          2             명사           A
2     이다          2             조사         NaN
3     사람          2             명사           A
4     서울          1          고유 명사           A
5     도심          1             명사           C
6   지나가다          1             동사           B
7     옛날          1             명사           A
8      물          1             명사           A
9     말다          1             동사           C
10    낚시          1             명사           B
11    하다          1             동사           A
12    살다          1             동사           A
13    기차          1             명사           A
14   차갑다          1            형용사           B
15    없다          1             동사         NaN
16     때          1             명사           A
17     배          1             명사           A
18    타고          1            NaN         NaN
19    다른          1            관형사           A
20    지역     

In [155]:
final_df = final_df.fillna({'topik_level': 'Поза словником', 'part_of_speech': 'Невідомо'})
print(final_df)

    word  frequency part_of_speech     topik_level
0     한강          5          고유 명사               A
1      강          2             명사               A
2     이다          2             조사  Поза словником
3     사람          2             명사               A
4     서울          1          고유 명사               A
5     도심          1             명사               C
6   지나가다          1             동사               B
7     옛날          1             명사               A
8      물          1             명사               A
9     말다          1             동사               C
10    낚시          1             명사               B
11    하다          1             동사               A
12    살다          1             동사               A
13    기차          1             명사               A
14   차갑다          1            형용사               B
15    없다          1             동사  Поза словником
16     때          1             명사               A
17     배          1             명사               A
18    타고          1       Невід

In [156]:
from deep_translator import MyMemoryTranslator

translator = MyMemoryTranslator(source='korean', target='english')

def safe_translate(text):
    if pd.isna(text):
        return text

    try:
        clean_text = str(text).strip()
        return translator.translate(clean_text)
    except Exception as e:
        return f"[Помилка: {e}]"

final_df['translation'] = final_df['word'].apply(safe_translate)

print(final_df)

    word  frequency part_of_speech     topik_level  \
0     한강          5          고유 명사               A   
1      강          2             명사               A   
2     이다          2             조사  Поза словником   
3     사람          2             명사               A   
4     서울          1          고유 명사               A   
5     도심          1             명사               C   
6   지나가다          1             동사               B   
7     옛날          1             명사               A   
8      물          1             명사               A   
9     말다          1             동사               C   
10    낚시          1             명사               B   
11    하다          1             동사               A   
12    살다          1             동사               A   
13    기차          1             명사               A   
14   차갑다          1            형용사               B   
15    없다          1             동사  Поза словником   
16     때          1             명사               A   
17     배          1         

In [157]:
final_df.to_csv('korean_words.csv', index=False, encoding='utf-8-sig')